# AMF Demonstration: Multi-Instance

**Purpose.** Reproduce the numbers reported in Section VII.F of the revised manuscript:

- Per-instance load-balance shares of the aggregate arrival rate Λ(t).
- Confirmation that a CPU-overload anomaly applied to AMF_01 is confined to that instance.
- Cross-instance Pearson correlations on `RES.CpuUtil`, normal vs. fault windows.

**Configuration.** N_AMF = 3, 14-day window, 100,000 UEs, seed = 42, all other parameters as in Table 6 of the manuscript.

**Output.** Reproduces the values reported in:

- Section VII.F: per-instance load shares, pairwise KS p-values on `RES.CpuUtil` (normal rows), and per-instance fault-window CPU statistics.
- Section VII.G: generation time and peak resident memory for the N_AMF=3 configuration.

In [1]:
# Imports
import numpy as np
import pandas as pd
import time, os, resource
from scipy.stats import ks_2samp, pearsonr

# --- Load AMFDatasetGenerator from the umbrella notebook ------------------
# The generator class is defined inside AMF_Pipeline_Complete.ipynb (cells
# 1-15: imports, constants, helper classes, AMFDatasetGenerator). We execute
# those cells here to bring the class into our namespace.
#
# Adjust UMBRELLA_NB_PATH if the file lives elsewhere.

import json

UMBRELLA_NB_PATH = 'AMF_Pipeline_Complete.ipynb'  # in same dir as this notebook
# If your umbrella notebook is in a parent directory, use '../AMF_Pipeline_Complete.ipynb'
# If it's in a subdirectory like 'notebooks/', use 'notebooks/AMF_Pipeline_Complete.ipynb'

with open(UMBRELLA_NB_PATH) as f:
    _umbrella = json.load(f)

# Cells 0-15 of the umbrella define everything the generator needs:
#   0: header (markdown)
#   1-9: imports, constants, utility functions
#   10: configuration block (EVENT_DATA, SLICE_*, SLICE_CPU_MULT etc.)
#   11: StatUtils
#   12: TemporalEngine
#   13: UEStateModel
#   14: AnomalyEngine helper function
#   15: AMFDatasetGenerator
# We execute through cell 15 to bring the class into this namespace.
for _cell in _umbrella['cells'][:16]:
    if _cell['cell_type'] == 'code':
        _code = ''.join(_cell['source']) if isinstance(_cell['source'], list) else _cell['source']
        # Skip cells whose first non-blank line is a Jupyter magic or shell command.
        # These (e.g. %%capture, !pip install) are not valid Python and would
        # raise SyntaxError when exec()'d.
        _first = next((ln for ln in _code.splitlines() if ln.strip()), '')
        if _first.startswith(('%', '!', '?')):
            continue
        try:
            exec(_code, globals())
        except Exception as _e:
            # Non-critical cells (e.g. optional package installs) may fail in
            # restricted environments. Print and continue — the import of
            # AMFDatasetGenerator below will catch any *critical* missing piece.
            print(f'  (Note: a cell in the umbrella raised {type(_e).__name__}; continuing.)')

# Verify the class is now available
assert 'AMFDatasetGenerator' in globals(), \
    'AMFDatasetGenerator class not loaded. Check UMBRELLA_NB_PATH.'
print('AMFDatasetGenerator class loaded successfully from', UMBRELLA_NB_PATH)


Mode: Generate | OUT_ROOT: /content/amf_pipeline_outputs
sdv installed
All imports OK.
GENERATE_FRESH=True — Kaggle download skipped.
AMFDatasetGenerator class loaded successfully from AMF_Pipeline_Complete.ipynb


In [2]:
# --- Configuration: 14-day, 3-instance, single CPU-overload anomaly on AMF_01 ---
# The generator accepts flat keyword arguments (see AMFDatasetGenerator.__init__).
# We split 100,000 UEs proportionally across slices: 70k eMBB / 20k mMTC / 10k URLLC.

CONFIG = dict(
    seed              = 42,
    amf_instances     = 3,           # <-- the only change vs. released SCOPE-5G Dataset (N_AMF=1)
    ue_embb           = 70_000,
    ue_mmtc           = 20_000,
    ue_urllc          = 10_000,
    duration_hours    = 14 * 24,     # 336 hours = 14 days
    step_min          = 15,
    vcpus_per_amf     = 8,
    mem_max_mb        = 8192,
    include_anomalies = True,
)

# Single scenario applied to AMF_01 (the second instance) for the anomaly-scoping test.
# Schema: each scenario is a dict with these fields (see build_anomaly_mask):
#   type, instance, day, start (HH:MM), duration_h, intensity, ramp_h
ANOMALY_SCHEDULE = [
    {"type": "cpu_overload", "instance": "AMF_01",
     "day": 5, "start": "09:00", "duration_h": 2.0,
     "intensity": "moderate", "ramp_h": 0.5},
]


In [3]:
# --- Run the generator and time it (Section VII.G numbers) ---
t0 = time.time()
gen = AMFDatasetGenerator(anomaly_scenarios=ANOMALY_SCHEDULE, **CONFIG)
df = gen.generate()
elapsed_s = time.time() - t0
peak_mem_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
peak_mem_mb = peak_mem_kb / 1024 if os.uname().sysname == 'Linux' else peak_mem_kb / (1024*1024)
print(f'[VII.G] N_AMF=3 generation time: {elapsed_s:.1f} s')
print(f'[VII.G] N_AMF=3 peak memory:    {peak_mem_mb:.0f} MB')
# 3 instances x 14 days x 24h x 4 slots/h = 4032 rows expected
print(f'        Output rows: {len(df)}, expected {3 * 14 * 24 * 4} = 4032')


[VII.G] N_AMF=3 generation time: 37.4 s
[VII.G] N_AMF=3 peak memory:    301 MB
        Output rows: 4032, expected 4032 = 4032


In [4]:
# --- Per-instance load balance (Section VII.F paragraph 2) ---
# Assuming the generator writes an 'amf_instance_id' column.
instances = sorted(df['amf_instance_id'].unique())
totals = df.groupby('amf_instance_id')['RM.RegReqAtt'].sum()
shares = totals / totals.sum() * 100
print('[VII.F] Per-instance load shares (%):')
for inst, share in shares.items():
    print(f'        {inst}: {share:.2f}%')
print('        Expected: roughly 1/3 each (within ±κ jitter U(0.8, 1.2))')

# Pairwise KS p-values on CPU utilization (normal rows only)
normal = df[df['is_anomaly'] == 0]
print('[VII.F] Pairwise KS test on RES.CpuUtil (normal rows only):')
for i in range(len(instances)):
    for j in range(i+1, len(instances)):
        a = normal[normal['amf_instance_id'] == instances[i]]['RES.CpuUtil']
        b = normal[normal['amf_instance_id'] == instances[j]]['RES.CpuUtil']
        stat, p = ks_2samp(a, b)
        print(f'        {instances[i]} vs {instances[j]}: p={p:.3f}')

[VII.F] Per-instance load shares (%):
        AMF_00: 31.81%
        AMF_01: 31.07%
        AMF_02: 37.11%
        Expected: roughly 1/3 each (within ±κ jitter U(0.8, 1.2))
[VII.F] Pairwise KS test on RES.CpuUtil (normal rows only):
        AMF_00 vs AMF_01: p=0.987
        AMF_00 vs AMF_02: p=0.963
        AMF_01 vs AMF_02: p=0.985


In [5]:
# --- Anomaly-scoping test (Section VII.F paragraph 3) ---
# A single CPU-overload on AMF_01 should affect only that instance.
# To demonstrate this we compare CPU at the fault timestamps across ALL three
# instances, not just rows where is_anomaly==1 (which by construction only
# flags AMF_01's rows and would leave the other two invisible).

fault_times = df[df['is_anomaly'] == 1]['timestamp'].unique()
normal = df[df['is_anomaly'] == 0]

print(f'[VII.F] Fault window: {len(fault_times)} timestamps')
print('[VII.F] CPU-utilization at fault timestamps, per instance:')
for inst in instances:
    rows = df[(df['amf_instance_id'] == inst) &
              (df['timestamp'].isin(fault_times))]
    cpu  = rows['RES.CpuUtil']
    base = normal[normal['amf_instance_id'] == inst]['RES.CpuUtil'].mean()
    print(f'        {inst}: fault-window mean={cpu.mean():.2f}%, '
          f'max={cpu.max():.2f}%, n={len(cpu)} '
          f'(normal baseline mean={base:.2f}%)')
print('        Only AMF_01 shows a meaningful rise above its baseline.')

[VII.F] Fault window: 8 timestamps
[VII.F] CPU-utilization at fault timestamps, per instance:
        AMF_00: fault-window mean=9.51%, max=12.02%, n=8 (normal baseline mean=8.68%)
        AMF_01: fault-window mean=12.70%, max=16.51%, n=8 (normal baseline mean=8.70%)
        AMF_02: fault-window mean=9.03%, max=11.29%, n=8 (normal baseline mean=8.68%)
        Only AMF_01 shows a meaningful rise above its baseline.


In [6]:
# --- Cross-instance correlation (Section VII.F paragraph 4) ---
pivot = (df.pivot_table(values='RES.CpuUtil', index='timestamp',
                         columns='amf_instance_id', aggfunc='first'))

print('[VII.F] Pairwise Pearson r on RES.CpuUtil (full window):')
for i in range(len(instances)):
    for j in range(i+1, len(instances)):
        r, _ = pearsonr(pivot[instances[i]].dropna(), pivot[instances[j]].dropna())
        print(f'        {instances[i]} <-> {instances[j]}: r={r:.3f}')

# Same restricted to the fault window — should drop because only AMF_01 is perturbed
fault_pivot = pivot.loc[df[df['is_anomaly'] == 1]['timestamp'].unique()]
print('[VII.F] Pairwise Pearson r on RES.CpuUtil (fault window only):')
for i in range(len(instances)):
    for j in range(i+1, len(instances)):
        if instances[i] in fault_pivot.columns and instances[j] in fault_pivot.columns:
            a = fault_pivot[instances[i]].dropna()
            b = fault_pivot[instances[j]].dropna()
            common = a.index.intersection(b.index)
            if len(common) > 2:
                r, _ = pearsonr(a.loc[common], b.loc[common])
                print(f'        {instances[i]} <-> {instances[j]}: r={r:.3f}')

[VII.F] Pairwise Pearson r on RES.CpuUtil (full window):
        AMF_00 <-> AMF_01: r=0.845
        AMF_00 <-> AMF_02: r=0.840
        AMF_01 <-> AMF_02: r=0.840
[VII.F] Pairwise Pearson r on RES.CpuUtil (fault window only):
        AMF_00 <-> AMF_01: r=0.445
        AMF_00 <-> AMF_02: r=0.221
        AMF_01 <-> AMF_02: r=0.766


## Reproducibility note

The values printed above are reported directly in §VII.F (per-instance load shares, pairwise KS p-values, per-instance fault-window CPU statistics) and §VII.G (N_AMF=3 generation time and peak memory) of the published manuscript. Re-running this notebook with `RANDOM_SEED=42` on a Google Colab standard CPU runtime reproduces the same output deterministically; small variations in peak memory are expected across machines.